# 12주차: 설명의 한계와 프라이버시

- CPU 실행, Python 표준 라이브러리만 사용, GPU·설치·모델 호출 불필요
- 작성한 응답 예시와 상정 참여 레코드 사용. 실제 모델 출력·개인자료 없음
- 수정할 곳: 노이즈 multiplier 숫자 하나. 변경 없이 전체 실행 가능
- 공개 제주 문서 원문·저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 원문 CC BY-SA 4.0, 실행 시 출처와 판 번호 출력, 원문 저장 없음
- 논문: https://arxiv.org/abs/2305.04388
- DP 원리: https://www.cis.upenn.edu/~aaroth/Papers/privacybook.pdf 2.3, 3.3, 3.5절


## 1. 공개 문서의 출처 확인

- 행정시 이름을 공개 문서에서 확인. 응답 예시는 원문 인용이 아닌 작성 예시
- 네트워크 실패는 미확인으로 출력. 예시 실행이 성공해도 공개 사실 확인 성공을 주장하지 않음


In [ ]:
import html, json, math, random, re, statistics, urllib.parse, urllib.request
OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
print("공개 문서 및 저자 목록:", SOURCE, "라이선스: CC BY-SA 4.0")
source_verified = False
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
try:
    request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
    with urllib.request.urlopen(request, timeout=30) as response:
        parsed = json.load(response)["parse"]
    assert parsed["revid"] == OLDID, "판 번호 불일치"
    source_text = html.unescape(re.sub(r"<[^>]+>", " ", parsed["text"]))
    assert "제주시" in source_text and "서귀포시" in source_text, "범주 이름 확인 실패"
    source_verified = True
    print("원문 판 번호 확인:", parsed["revid"], "범주 이름: 제주시, 서귀포시")
    del source_text, parsed
except (OSError, ValueError, KeyError, AssertionError) as error:
    print("공개 문서 조회 미확인:", type(error).__name__)
print("이후 자료: 교수 작성 응답과 상정 레코드, 실제 모델·개인 응답 아님")


## 2. 서술-결과 불일치 프록시

- 설명에서 주장한 수와 최종 답의 수만 비교하는 좁은 규칙
- reference_count=2는 교육 예제의 정답 기준, 최신 행정 변화를 추정하는 모델 아님
- 내부 계산 접근 없음. 모든 예시의 내부 충실성은 미측정


In [ ]:
reference_count = 2
examples = [
    {"id": "A", "reason_text": "제주시와 서귀포시, 행정시 수는 2.", "reason_count": 2, "answer_count": 2},
    {"id": "B", "reason_text": "행정시 수는 3.", "reason_count": 3, "answer_count": 2},
    {"id": "C", "reason_text": "제주시와 서귀포시, 행정시 수는 2.", "reason_count": 2, "answer_count": 3},
    {"id": "D", "reason_text": "행정시 수는 3.", "reason_count": 3, "answer_count": 3},
]
print("작성 예시 | 설명 수 | 답 수 | 설명 수 정확 | 답 정확 | 불일치 | 내부 충실성")
for e in examples:
    e["mismatch"] = e["reason_count"] != e["answer_count"]
    print(e["id"], e["reason_count"], e["answer_count"],
          e["reason_count"] == reference_count, e["answer_count"] == reference_count,
          e["mismatch"], "미측정", sep=" | ")
print("불일치:", sum(e["mismatch"] for e in examples), "/", len(examples))
print("B: 정답이어도 설명 불일치. D: 불일치 없어도 오답. 내부 faithfulness 측정값 아님.")


## 3. 상정 통계와 민감도

- 공개 행정시 이름을 범주로 사용하는 작성 선호 레코드 20개
- 실제 관광 조사, 주민 의견, 학생 정보 아님
- 인접 관계: 레코드 하나 추가/제거(add/remove), 한 명의 기여 최대 1
- 공개 질의: 제주시를 선택한 count 하나. 전역 민감도 Δf=1


In [ ]:
records = ["제주시"] * 12 + ["서귀포시"] * 8
true_count = sum(place == "제주시" for place in records)
sensitivity = 1.0
base_epsilon = 1.0
print("상정 레코드 수:", len(records), "제주시 count:", true_count)
for added in ("제주시", "서귀포시"):
    neighbor = records + [added]
    neighbor_count = sum(place == "제주시" for place in neighbor)
    print("하나 추가:", added, "count 변화:", abs(neighbor_count - true_count))
print("한 레코드의 0/1 기여 제한으로 민감도 1. 실제 참여 자료 없음.")


## 4. 한 숫자 변경

- 먼저 기본값 그대로 전체 실행. 다음에 1.0을 2.0 또는 3.0으로 변경하고 전체 실행
- scale = multiplier × Δf / base_epsilon
- 이상적인 메커니즘의 유효 epsilon = base_epsilon / multiplier
- 공개 seed는 재현성용. 실제 민감한 데이터의 DP 공개에는 사용할 수 없음


In [ ]:
# TODO: 이 숫자만 2.0 또는 3.0으로 변경
NOISE_MULTIPLIER = 1.0


In [ ]:
assert math.isfinite(NOISE_MULTIPLIER) and NOISE_MULTIPLIER > 0, "양의 유한한 숫자 사용"
scale = NOISE_MULTIPLIER * sensitivity / base_epsilon
effective_epsilon = sensitivity / scale
print("scale:", scale, "이상적 메커니즘 유효 epsilon:", effective_epsilon)
print("보장 범위 주의: 공개 seed 교육 실험, 실제 DP 공개 보장 없음")


## 5. 같은 표본으로 scale 변화 관찰

- 표준 Laplace 표본 z에 scale b를 곱해 noise=bz 생성
- 같은 z로 scale만 비교하므로 평균 절대 오차는 b에 정확히 비례
- 이론적 E|noise|=b, 표본 평균은 근사값
- 음수를 0으로 바꾸는 clipping은 공개 출력만 쓰는 사후 처리
- 이 노트북의 2,000회 반복은 상정 데이터 오차 관찰. 실제 개인 통계를 2,000회 공개하는 계획 아님


In [ ]:
rng = random.Random(202612)
def standard_laplace():
    # 열린 구간의 교육용 inverse-CDF 표본. 운영용 안전한 DP 구현 아님.
    u = 0.0
    while u == 0.0:
        u = rng.random()
    return math.log(2 * u) if u < 0.5 else -math.log(2 * (1 - u))

standard_noise = [standard_laplace() for _ in range(2000)]
print("multiplier | scale | 유효 epsilon | 평균절대오차 | 음수출력비율 | clipping후오차")
for multiplier in dict.fromkeys([1.0, NOISE_MULTIPLIER]):
    b = multiplier * sensitivity / base_epsilon
    released = [true_count + b * z for z in standard_noise]
    mae = statistics.mean(abs(value - true_count) for value in released)
    clipped = [max(0.0, value) for value in released]
    clipped_mae = statistics.mean(abs(value - true_count) for value in clipped)
    negative_rate = sum(value < 0 for value in released) / len(released)
    print(f"{multiplier:.1f} | {b:.1f} | {sensitivity / b:.3f} | {mae:.3f} | {negative_rate:.3f} | {clipped_mae:.3f}")
print("표준표본 평균 절댓값:", round(statistics.mean(abs(z) for z in standard_noise), 6))
print("같은 표본 사용: 변경 전후 clipping 전 MAE 비율 = multiplier 비율")


## 6. 반복 공개와 판단 보류

- 이상적인 epsilon-DP 독립 공개 k회의 기본 순차 합성 상한: k × epsilon
- 여러 응답의 평균을 공개한다고 예산이 사라지지 않음
- 공개 seed로 계산한 아래 숫자는 예산 산술 관찰이며 보호된 서비스가 아님
- 실제 프로젝트는 노이즈를 더하기 전에 수집하지 않을 자료와 사람 확인 경계를 먼저 설정


In [ ]:
for k in (1, 5, 10):
    print("공개 횟수:", k, "기본 순차 합성 epsilon 상한:", round(k * effective_epsilon, 3))
print("허용 예시: 공개 문서 출처와 확인 시점 안내")
print("보류 예시: 안전·자격·권리 최종 판정, 지역 정책 수용성 자동 확정")
print("출처 확인 상태:", "확인" if source_verified else "미확인")
print("완료: 내부 충실성 미측정, 실제 개인정보 없음, 실제 DP 서비스 보장 없음")


## 결과 해석 기록

- B와 D의 차이: [불일치 여부]와 [정답 여부]를 다른 문장으로 설명
- multiplier [값], scale [값], 유효 epsilon [값], 관찰 MAE [값]
- 공개 seed·실제 기여 제한·반복 공개 예산 중 운영 전 바꿀 조건 두 가지 기록
- 강의의 위험 체크리스트와 설계도 §선 양식을 팀 목적에 맞게 작성


## 돌아보기: 확인 질문

1. B와 D의 설명-결과 불일치와 최종 답 정확성을 서로 다른 문장으로 기록
2. multiplier 1과 3의 scale·유효 epsilon·관찰 MAE를 출력과 대조
3. 공개 seed의 재현 가능한 교육 실험이 실제 DP 공개 보장이 아닌 이유 설명
4. 프로젝트 체크리스트에서 사용 목적·사람 확인·미확인 근거를 구분

## 제출 안내

- 코드 구현이나 자료 수집을 새로 요구하는 과제가 아닌 제공된 실습의 관찰 기록
- 별도 제출 공지가 있을 때 실행한 노트북의 출력 표와 질문별 관찰·판정 이유를 함께 제출
- 저장된 결과에서 미확인·미판정은 그대로 표시, 모델 학습 성과로 보고하지 않는 원칙
- 학생 폴더에는 학번만 사용, 이름·연락처·데이터 원본·API 키 제외
